In [ ]:
# Paquetes esenciales
!pip install opencv-python-headless
!pip install ninja
!pip install smdebug

In [ ]:
# Herramientas de evaluación
!pip install torch-fidelity pillow

In [ ]:
# PyTorch con CUDA 
!pip install torch==2.0.1+cu117 torchvision==0.15.2+cu117 -f https://download.pytorch.org/whl/cu117/torch_stable.html

In [ ]:
import image_process_hr
from sagemaker.pytorch import PyTorch
import sagemaker
import torch_fidelity

## Configuración de descarga y Procesamiento de imágenes

Descarga y procesa imagenes una a una (eliminando la que ha sido subida luego de procesada)

In [ ]:
%%time

image_process_hr.get_and_process_images(
    s3_folder="computer-vision/dermatologia/train/nevus/",
    local_download_folder="img/prueba4/", ##Carpeta de salida temporal
    target_size=(256, 256),
    hr=True, #hr = False ## No realiza remocion de bellos           
    muestra=0, # si es 0 carga TODO dentro del directorio
    crop=True, #Recorta imágenes de acuerdo a su dimensión más corta
    resize_quality='ultra'  # ('basic', 'high', 'ultra')
)

## DataSet Tool

Ejecuta dataset_tool.py


In [ ]:
%reset -f

In [ ]:
dataset_folder = "/home/sagemaker-user/stylegan/img/prueba3/processed"
dataset_folder

In [ ]:
dataset_name = "melanoma_2_256x256" 
output_path = f"stylegan2-ada-pytorch-main/datasets/{dataset_name}.zip"

In [ ]:
!python /home/sagemaker-user/stylegan/stylegan2-ada-pytorch-main/dataset_tool.py --source={dataset_folder} --dest={output_path} --width=256 --height=256

Borrar imagenes despues del dataset para optimizar memoria, esto es opcional

In [ ]:
##No borrar hasta que se terminen de realizar las evaluaciones
'''
import shutil
from pathlib import Path
path = Path(dataset_folder)
if path.exists():
    shutil.rmtree(path)
'''

## Parametrizar modelo

El hiperparametro salida es el nombre del directorio en S3 que se guardan los eventos

In [ ]:
hps= {
    'data':f'datasets/{dataset_name}.zip',
    'outdir':'salida',
    'gpus':'1',
    'kimg':'2500',
    'batch':'32',
    'snap':'50',
    'salida': 'prueba3' ##Directorio de salida en S3 donde se guardan los eventos y archivos del modelo
}

my_estimator = PyTorch(role=sagemaker.get_execution_role(),
                        entry_point='train.py',
                        source_dir='stylegan2-ada-pytorch-main',
                        instance_count=1,
                        instance_type='ml.g5.2xlarge',
                        use_spot_instances=True,
                        max_run=172800,
                        max_wait=172800,
                        framework_version='1.8',
                        py_version='py36',
                        output_path='s3://avedian-ml/stylegan/train',
                        hyperparameters=hps
                      )

In [ ]:
%%time
my_estimator.fit()

In [ ]:
##Verificar si la instancia es GPU
!nvidia-smi

In [ ]:
##Eliminar cache de torch
rm -rf ~/.cache/torch_extensions

## Inferencias en el modelo

In [ ]:
!pwd

In [ ]:
##Copiar en la instancia el archivo pkl

#!aws s3 cp s3://avedian-ml/stylegan/melanoma/prueba3/completed_20250305_193910/00000-melanoma_2_256x256-auto1-kimg2500-batch32-resumecustom/network-snapshot-002500.pkl /home/sagemaker-user/stylegan/stylegan2-ada-pytorch-main


In [ ]:
%%time

##Si se quiere ejecutar con GPU, modificar en generate.py --> device = torch.device('cuda')

#!python /home/sagemaker-user/stylegan/stylegan2-ada-pytorch-main/generate.py --outdir=out --projected-w=out/projected_w.npz \
#    --network=network-snapshot-001000.pkl


!python /home/sagemaker-user/stylegan/stylegan2-ada-pytorch-main/generate.py --outdir=out/melanoma/train --network=stylegan2-ada-pytorch-main/network-snapshot-002500.pkl --seeds=2000-9999 2>/dev/null

**Evaluación**

In [ ]:
%%time
!python stylegan-evaluation.py --real /home/sagemaker-user/stylegan/img/prueba3/processed --fake /home/sagemaker-user/stylegan/out/2500/ --metrics isc,fid --max-images 2000

## **Evaluation Results:**

Imagenes generadas 2000, pkl 2500

inception_score_mean: 1.0890777111053467
inception_score_std: 0.004081445746123791
frechet_inception_distance: 63.44689233645897

## **Evaluation Results:**

Imagenes generadas 2000, pkl 2400

inception_score_mean: 1.0907747745513916
inception_score_std: 0.0031479261815547943
frechet_inception_distance: 63.8617586084693

## **Evaluation Results:**

Imagenes generadas 2000, pkl 1800

inception_score_mean: 1.0881567001342773
inception_score_std: 0.002997073344886303
frechet_inception_distance: 75.93674106172239

## Encontrar Imágenes similares

In [ ]:
!python find-similar-images.py --real /home/sagemaker-user/stylegan/img/prueba4/processed --fake /home/sagemaker-user/stylegan/out/nevus/train/ --top-n 20 --plot --max-images 10000